# Semantic Segmentation — Pascal VOC 2012 (Colab)

Entraîne un modèle du package `semseg` sur GPU, puis ouvre le dashboard.

*Runtime → Change runtime type → GPU.*

In [ ]:
!git clone https://github.com/corentin-ne/semantic_segmentation.git
%cd semantic_segmentation
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

## (Optionnel) sauvegarder les runs sur Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
RUNS = '/content/gdrive/MyDrive/semantic_segmentation/runs'

## Entraînement

`configs/deeplabv3plus_mobilenet.yaml` est le meilleur compromis. Le split `trainaug` télécharge aussi SBD (~1.4 Go) ; utilise `data.train_split=train` pour aller plus vite.

In [ ]:
!python -m semseg train -c configs/deeplabv3plus_mobilenet.yaml -o logging.runs_dir=$RUNS train.compile=true

## Évaluation, prédiction, export

In [ ]:
import glob
run = sorted(glob.glob(f'{RUNS}/*'))[-1]
!python -m semseg evaluate --run "$run"
!python -m semseg benchmark --device cuda --output $RUNS/benchmark_gpu.json
!python -m semseg export --run "$run"

## Dashboard (dans Colab)

In [ ]:
import subprocess
subprocess.Popen(['streamlit', 'run', 'dashboard/app.py', '--server.headless', 'true', '--', '--runs', RUNS])
from google.colab import output
output.serve_kernel_port_as_window(8501)